# Dancing Numbers RAG Pipeline v2
Run cells top to bottom.

In [1]:
import os
FOLDER = r"C:\Users\Kartik\Desktop\AI agent Chat"  # EDIT
os.chdir(FOLDER)
print("Working dir:", os.getcwd())
for f in ["blogs.db","blogs_faiss.index","chunks_map.npy"]:
    status = "EXISTS" if os.path.exists(f) else "MISSING"
    print(f"  {f:30s} {status}")


Working dir: C:\Users\Kartik\Desktop\AI agent Chat
  blogs.db                       EXISTS
  blogs_faiss.index              EXISTS
  chunks_map.npy                 EXISTS


In [2]:
import subprocess, sys
subprocess.check_call([sys.executable,"-m","pip","install",
    "requests","beautifulsoup4","sentence-transformers","faiss-cpu",
    "langchain","langchain-text-splitters","tqdm","lxml","--quiet"])
print("OK")


OK


In [3]:
import requests, sqlite3, time, re, warnings
import numpy as np
from bs4 import BeautifulSoup
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
try:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
except ImportError:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss
warnings.filterwarnings("ignore")
print("Imports OK")


Imports OK


In [4]:
SITEMAP_INDEX_URL = "https://www.dancingnumbers.com/sitemap_index.xml"
DB_PATH           = "blogs.db"
FAISS_INDEX_PATH  = "blogs_faiss.index"
CHUNK_MAP_PATH    = "chunks_map.npy"
EMBEDDING_MODEL   = "all-MiniLM-L6-v2"
CHUNK_SIZE        = 1200   # ~300 tokens x 4 chars
CHUNK_OVERLAP     = 200    # ~50 tokens x 4 chars
BATCH_SIZE        = 64
MAX_SITEMAPS      = 5
MAX_BLOG_URLS     = 50
CRAWL_DELAY       = 1.5
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
print("Config ready")


Config ready


## Step 1 - Upgrade SQLite Schema

In [5]:
conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()
cur.execute("""
    CREATE TABLE IF NOT EXISTS blogs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        url TEXT UNIQUE NOT NULL, title TEXT, content TEXT)""")
cur.execute("""
    CREATE TABLE IF NOT EXISTS chunks (
        chunk_id INTEGER PRIMARY KEY AUTOINCREMENT,
        blog_id  INTEGER NOT NULL,
        chunk_text TEXT NOT NULL,
        h1 TEXT DEFAULT "",
        FOREIGN KEY (blog_id) REFERENCES blogs(id))""")
try:
    cur.execute("ALTER TABLE chunks ADD COLUMN h1 TEXT DEFAULT \"\"  ")
    print("Added h1 column")
except sqlite3.OperationalError:
    print("h1 column already present")
cur.execute("""
    CREATE VIRTUAL TABLE IF NOT EXISTS chunks_fts
    USING fts5(chunk_text, content='chunks', content_rowid='chunk_id',
               tokenize='porter ascii')""")
conn.commit()
conn.close()
print("Schema ready")


h1 column already present
Schema ready


## Step 2 - Semantic Chunker Function

In [6]:
def fetch_url(url, timeout=15):
    try:
        r = requests.get(url, headers=HEADERS, timeout=timeout)
        r.raise_for_status()
        return r.text
    except Exception as e:
        return None

def extract_heading_chunks(html, url):
    """Split HTML into semantic chunks respecting H1/H2/H3 boundaries."""
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup.find_all(["nav","footer","aside","script","style","header"]):
        tag.decompose()
    body = None
    for sel in ["article","div.entry-content","div.post-content","main","body"]:
        body = soup.select_one(sel)
        if body: break
    if not body: body = soup

    h1_tag  = soup.find("h1")
    h1_text = h1_tag.get_text(strip=True) if h1_tag else ""

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n","\n",". "," ",""])

    sections, current_heading, current_parts = [], h1_text, []

    for tag in body.find_all(["h1","h2","h3","p","li","ul","ol"]):
        name = tag.name.lower()
        text = re.sub(r"\s+", " ", tag.get_text(separator=" ", strip=True))
        if not text or len(text) < 10: continue
        if name in {"h1","h2","h3"}:
            if current_parts:
                sections.append((current_heading, " ".join(current_parts)))
                current_parts = []
            current_heading = text
        elif len(text) > 20:
            current_parts.append(text)

    if current_parts:
        sections.append((current_heading, " ".join(current_parts)))

    result = []
    for heading, content in sections:
        if len(content) < 80: continue
        for chunk in splitter.split_text(content):
            chunk = chunk.strip()
            if len(chunk.split()) >= 10:
                result.append((h1_text or heading, chunk))
    return result

print("Chunker ready")


Chunker ready


## Step 3 - Scrape Blogs (skips if already done)

In [7]:
conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()
cur.execute("SELECT COUNT(*) FROM blogs WHERE content IS NOT NULL")
scraped = cur.fetchone()[0]
conn.close()

if scraped > 0:
    print(f"Already scraped {scraped} blogs. Skipping.")
else:
    xml = fetch_url(SITEMAP_INDEX_URL)
    soup = BeautifulSoup(xml, "lxml-xml")
    sitemap_urls = [l.get_text(strip=True) for l in soup.find_all("loc")]
    all_blog_urls = []
    for sm in tqdm(sitemap_urls[:MAX_SITEMAPS], desc="Sitemaps"):
        x = fetch_url(sm)
        if not x: continue
        locs = [l.get_text(strip=True) for l in BeautifulSoup(x,"lxml-xml").find_all("loc")]
        all_blog_urls.extend([u for u in locs
            if u.startswith("https://www.dancingnumbers.com/")
            and not any(s in u for s in ["/tag/","/category/","?p=","/page/","/author/"])
            and u != "https://www.dancingnumbers.com/"])
        time.sleep(CRAWL_DELAY)
    all_blog_urls = list(dict.fromkeys(all_blog_urls))[:MAX_BLOG_URLS]
    conn = sqlite3.connect(DB_PATH)
    cur  = conn.cursor()
    for url in all_blog_urls:
        cur.execute("INSERT OR IGNORE INTO blogs (url) VALUES (?)", (url,))
    conn.commit()
    cur.execute("SELECT id, url FROM blogs WHERE content IS NULL")
    rows = cur.fetchall()
    for blog_id, url in tqdm(rows, desc="Scraping"):
        html = fetch_url(url)
        if not html: continue
        s = BeautifulSoup(html, "html.parser")
        h = s.find("h1")
        title = h.get_text(strip=True) if h else (s.title.get_text(strip=True) if s.title else "")
        content = re.sub(r"\s+", " ", s.get_text(separator=" ", strip=True)).strip()
        if len(content) >= 100:
            cur.execute("UPDATE blogs SET title=?, content=? WHERE id=?", (title,content,blog_id))
        conn.commit()
        time.sleep(CRAWL_DELAY)
    conn.close()
    print("Scraping done")


Already scraped 851 blogs. Skipping.


## Step 4 - Semantic Re-Chunking

In [8]:
conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()
cur.execute("DELETE FROM chunks")
cur.execute("DELETE FROM chunks_fts")
conn.commit()
cur.execute("SELECT id, url, content FROM blogs WHERE content IS NOT NULL")
rows = cur.fetchall()
print(f"Re-chunking {len(rows)} blogs...")
total = 0
for blog_id, url, content in tqdm(rows, desc="Chunking"):
    html = fetch_url(url)
    if html:
        chunks = extract_heading_chunks(html, url)
    else:
        sp = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
        chunks = [("", c) for c in sp.split_text(content)]
    time.sleep(0.5)
    for h1_text, chunk_text in chunks:
        if len(chunk_text.split()) < 10: continue
        cur.execute("INSERT INTO chunks (blog_id, chunk_text, h1) VALUES (?,?,?)",
                    (blog_id, chunk_text.strip(), h1_text))
        total += 1
conn.commit()
print(f"Rebuilding FTS5 index...")
cur.execute("INSERT INTO chunks_fts(rowid, chunk_text) SELECT chunk_id, chunk_text FROM chunks")
conn.commit()
conn.close()
print(f"Done - {total:,} chunks | FTS5 index built")


Re-chunking 851 blogs...


Chunking: 100%|██████████| 851/851 [32:36<00:00,  2.30s/it]


Rebuilding FTS5 index...
Done - 17,513 chunks | FTS5 index built


## Step 5 - Generate Normalized Embeddings

In [9]:
print(f"Loading: {EMBEDDING_MODEL}")
model = SentenceTransformer(EMBEDDING_MODEL)
conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()
cur.execute("SELECT chunk_id, chunk_text FROM chunks ORDER BY chunk_id")
rows = cur.fetchall()
conn.close()
chunk_ids   = [r[0] for r in rows]
chunk_texts = [r[1] for r in rows]
print(f"Embedding {len(chunk_texts):,} chunks...")
embeddings = model.encode(
    chunk_texts, batch_size=BATCH_SIZE, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True
).astype(np.float32)
# Explicit L2 normalization
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
norms = np.where(norms == 0, 1, norms)
embeddings = embeddings / norms
print(f"Shape: {embeddings.shape}")
print(f"Norm check (should be 1.0): {np.linalg.norm(embeddings[0]):.6f}")


Loading: all-MiniLM-L6-v2
Embedding 17,513 chunks...


Batches:   0%|          | 0/274 [00:00<?, ?it/s]

Shape: (17513, 384)
Norm check (should be 1.0): 1.000000


## Step 6 - Build FAISS IndexFlatIP

In [10]:
dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # cosine sim on normalized vectors
index.add(embeddings)
faiss.write_index(index, FAISS_INDEX_PATH)
np.save(CHUNK_MAP_PATH, np.array(chunk_ids))
print(f"FAISS IndexFlatIP - {index.ntotal:,} vectors")
for f in [FAISS_INDEX_PATH, CHUNK_MAP_PATH, DB_PATH]:
    size = os.path.getsize(os.path.abspath(f)) / 1024
    print(f"  {f:30s}  {size:,.0f} KB")


FAISS IndexFlatIP - 17,513 vectors
  blogs_faiss.index               26,270 KB
  chunks_map.npy                  137 KB
  blogs.db                        34,008 KB


## Step 7 - Verify & Install Reranker

In [11]:
subprocess.check_call([sys.executable,"-m","pip","install","sentence-transformers","--quiet"])
from sentence_transformers import CrossEncoder
print("Installing reranker (first run downloads ~80 MB)...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512)
print("Reranker loaded OK")


Installing reranker (first run downloads ~80 MB)...
Reranker loaded OK


In [12]:
# Final test
test_q = "how to fix payment issue in quickbooks"
q_emb  = model.encode([test_q], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
scores, indices = index.search(q_emb, 5)
id_map = np.load(CHUNK_MAP_PATH)
conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()
print(f"Test: \"{test_q}\"\n")
for rank, (sc, idx) in enumerate(zip(scores[0], indices[0]), 1):
    cid = int(id_map[idx])
    cur.execute("""SELECT c.h1, c.chunk_text, b.title, b.url
        FROM chunks c JOIN blogs b ON c.blog_id = b.id WHERE c.chunk_id = ?""", (cid,))
    row = cur.fetchone()
    if row:
        print(f"Rank {rank} | Score: {sc:.4f} | H1: {row[0][:50]}")
        print(f"  {row[1][:120]}...")
        print()
conn.close()
print("Pipeline v2 complete! Run: streamlit run app.py")


Test: "how to fix payment issue in quickbooks"

Rank 1 | Score: 0.8497 | H1: How to Fix QuickBooks Payment Error Due To Interne
  Follow the below mention steps to fix to QuickBooks Payment Error Due to Internet Connection....

Rank 2 | Score: 0.8191 | H1: How to Create an Invoice in QuickBooks Desktop?
  Open QuickBooks Desktop. Click on the Customers menu. Then click on Receive Payment. Choose the name of the customer by ...

Rank 3 | Score: 0.8033 | H1: QuickBooks Generated Zero Amount Transaction for B
  There can also be scenarios when the credit payment or vendor is displayed as unapplied. In such scenarios, you can reso...

Rank 4 | Score: 0.7793 | H1: How to Receive Payments in QuickBooks Online
  While learning how to receive payments in QuickBooks Online, watch out for some common mistakes. Here are a few issues t...

Rank 5 | Score: 0.7791 | H1: How to Troubleshoot QuickBooks Payments (Merchant 
  QuickBooks Payment often fails to work due to old software, a damaged company 